# Agents on Model Serving — LangGraph Example

This notebook demonstrates how to build, test, and deploy a **LangGraph ReAct agent** as a **Databricks Model Serving endpoint** using the [Agent Framework](https://docs.databricks.com/en/generative-ai/agent-framework/author-agent-model-serving.html).

**What's covered:**
- Define a single tool and wire it into a LangGraph `create_react_agent` graph
- Wrap the graph in an MLflow `ResponsesAgent` with both **streaming** and **non-streaming** support
- Log & register the agent to **Unity Catalog**
- Deploy to a **Model Serving endpoint** (AI Playground compatible)
- Invoke the live endpoint via REST

**Model:** GPT 5.4 via Foundation Model API &nbsp;|&nbsp; **Framework:** LangGraph + `databricks-langchain`

---

**References / Credits:**
- [LangGraph MCP Tool-Calling Agent Notebook](https://docs.databricks.com/aws/en/notebooks/source/generative-ai/langgraph-mcp-tool-calling-agent.html)
- [MLflow ResponsesAgent for Model Serving](https://mlflow.org/docs/latest/genai/serving/responses-agent)
- [Author an AI Agent and Deploy on Model Serving](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent-model-serving?language=LangChain%2FLangGraph)

In [0]:
%pip install -U -qqq databricks-agents mlflow databricks-langchain langgraph nest_asyncio
dbutils.library.restartPython()

In [0]:
# --- Configuration (update these if needed) ---
CATALOG_NAME = "ram-agents"
SCHEMA_NAME = "langgraph_serving"
MODEL_NAME = "langgraph_agent"

# --- Create catalog and schema if they don't exist ---
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG_NAME}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.`{SCHEMA_NAME}`")

# Fully qualified UC model path (used by logging, registration, and deployment cells)
UC_MODEL_NAME = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{MODEL_NAME}`"

print(f"Catalog : {CATALOG_NAME}")
print(f"Schema  : {SCHEMA_NAME}")
print(f"UC Model: {UC_MODEL_NAME}")

In [0]:
import mlflow
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from databricks_langchain import ChatDatabricks

# --- Define a simple tool ---
@tool
def lookup_databricks_docs(query: str) -> str:
    """Look up information about Databricks products and features.
    Use this tool when the user asks about Databricks, its products, or capabilities."""
    knowledge_base = {
        "unity catalog": "Unity Catalog provides centralized governance for data and AI assets across Databricks workspaces, supporting fine-grained access control, lineage tracking, and data discovery.",
        "model serving": "Databricks Model Serving provides production-grade, scalable REST API endpoints for deploying ML models and AI agents with automatic scaling and monitoring.",
        "delta lake": "Delta Lake is an open-source storage layer that brings ACID transactions, schema enforcement, and time travel to data lakes built on Apache Spark.",
        "mlflow": "MLflow is an open-source platform for managing the end-to-end machine learning lifecycle, including experiment tracking, model packaging, and deployment.",
    }
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return value
    return (
        f"Databricks is a unified analytics platform. For specific information about "
        f"'{query}', please refer to the official Databricks documentation."
    )

# --- Configure LLM via Foundation Model API ---
# Update the endpoint name if your workspace uses a different FM API endpoint for this model
FM_API_ENDPOINT = "databricks-gpt-5-4"
llm = ChatDatabricks(endpoint=FM_API_ENDPOINT)

# --- Build LangGraph ReAct agent ---
tools = [lookup_databricks_docs]
langgraph_agent = create_react_agent(llm, tools=tools)

print(f"LangGraph agent created with endpoint: {FM_API_ENDPOINT}")
print(f"Tools: {[t.name for t in tools]}")

In [0]:
import asyncio
import json
from typing import AsyncGenerator, Generator

import nest_asyncio
from langchain.messages import AIMessageChunk
from langchain_core.messages.tool import ToolMessage
from langgraph.graph.state import CompiledStateGraph
from mlflow.models import set_model
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)

# Allow nested event loops (required in notebook & model serving environments)
nest_asyncio.apply()


class LangGraphResponsesAgent(ResponsesAgent):
    """Wraps a compiled LangGraph agent as a ResponsesAgent for Model Serving deployment.
    Supports both streaming (predict_stream) and non-streaming (predict) invocation.

    Streaming uses two LangGraph stream modes:
      - "updates":  complete node outputs (tool calls, tool results, final messages)
      - "messages": token-level AIMessageChunks for real-time text deltas in AI Playground
    """

    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        """Non-streaming: collects all done-items from the stream into a single response."""
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done" or event.type == "error"
        ]
        return ResponsesAgentResponse(output=outputs)

    async def _predict_stream_async(
        self, request: ResponsesAgentRequest
    ) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
        """Async streaming core — yields both node-level updates and token-level deltas."""
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])

        async for event in self.agent.astream(
            {"messages": cc_msgs}, stream_mode=["updates", "messages"]
        ):
            if event[0] == "updates":
                # Complete node outputs: tool calls, tool results, final assistant messages
                for node_data in event[1].values():
                    msgs = node_data.get("messages", [])
                    if msgs:
                        # Ensure ToolMessage content is always a string
                        for msg in msgs:
                            if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                                msg.content = json.dumps(msg.content)
                        for item in output_to_responses_items_stream(msgs):
                            yield item

            elif event[0] == "messages":
                # Token-level chunks — powers real-time typing in AI Playground
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except Exception:
                    pass

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Sync wrapper over the async stream for MLflow / Model Serving compatibility."""
        agen = self._predict_stream_async(request)

        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)

        ait = agen.__aiter__()
        while True:
            try:
                item = loop.run_until_complete(ait.__anext__())
            except StopAsyncIteration:
                break
            else:
                yield item


# Enable MLflow autologging for LangChain/LangGraph traces
mlflow.langchain.autolog()

# Instantiate and register the agent
agent = LangGraphResponsesAgent(langgraph_agent)
set_model(agent)

print("ResponsesAgent registered via set_model(). Ready for testing and logging.")

In [0]:
# Test non-streaming (predict)
response = agent.predict(
    {"input": [{"role": "user", "content": "What is Unity Catalog in Databricks?"}]}
)

print("=== Non-Streaming Response ===")
for item in response.output:
    print(f"  Type: {item.type}")
    if hasattr(item, "text"):
        print(f"  Text: {item.text}")
    elif hasattr(item, "name"):
        print(f"  Tool: {item.name} | Args: {item.arguments}")
    print("  ---")

In [0]:
# Test streaming (predict_stream)
print("=== Streaming Response ===")
for event in agent.predict_stream(
    {"input": [{"role": "user", "content": "Tell me about Delta Lake on Databricks"}]}
):
    print(f"  Event: {event.type}")
    if event.type == "response.output_item.done" and hasattr(event.item, "text"):
        print(f"  Text: {event.item.text}")
    elif event.type == "response.output_text.delta":
        print(f"  Delta: {event.delta}")
    print("  ---")

In [0]:
import pathlib
from mlflow.models.resources import DatabricksServingEndpoint

mlflow.set_registry_uri("databricks-uc")

# Write self-contained agent code to a file for code-based logging
agent_code = f'''
import asyncio
import json
from typing import AsyncGenerator, Generator

import mlflow
import nest_asyncio
from langchain.messages import AIMessageChunk
from langchain_core.messages.tool import ToolMessage
from langchain_core.tools import tool
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import create_react_agent
from databricks_langchain import ChatDatabricks
from mlflow.models import set_model
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)

nest_asyncio.apply()

# --- Tool ---
@tool
def lookup_databricks_docs(query: str) -> str:
    """Look up information about Databricks products and features.
    Use this tool when the user asks about Databricks, its products, or capabilities."""
    knowledge_base = {{
        "unity catalog": "Unity Catalog provides centralized governance for data and AI assets across Databricks workspaces, supporting fine-grained access control, lineage tracking, and data discovery.",
        "model serving": "Databricks Model Serving provides production-grade, scalable REST API endpoints for deploying ML models and AI agents with automatic scaling and monitoring.",
        "delta lake": "Delta Lake is an open-source storage layer that brings ACID transactions, schema enforcement, and time travel to data lakes built on Apache Spark.",
        "mlflow": "MLflow is an open-source platform for managing the end-to-end machine learning lifecycle, including experiment tracking, model packaging, and deployment.",
    }}
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return value
    return (
        "Databricks is a unified analytics platform. For specific information about "
        "'" + query + "', please refer to the official Databricks documentation."
    )

# --- LLM + Agent ---
llm = ChatDatabricks(endpoint="{FM_API_ENDPOINT}")
langgraph_agent = create_react_agent(llm, tools=[lookup_databricks_docs])


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done" or event.type == "error"
        ]
        return ResponsesAgentResponse(output=outputs)

    async def _predict_stream_async(
        self, request: ResponsesAgentRequest
    ) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        async for event in self.agent.astream(
            {{"messages": cc_msgs}}, stream_mode=["updates", "messages"]
        ):
            if event[0] == "updates":
                for node_data in event[1].values():
                    msgs = node_data.get("messages", [])
                    if msgs:
                        for msg in msgs:
                            if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                                msg.content = json.dumps(msg.content)
                        for item in output_to_responses_items_stream(msgs):
                            yield item
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except Exception:
                    pass

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        agen = self._predict_stream_async(request)
        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
        ait = agen.__aiter__()
        while True:
            try:
                item = loop.run_until_complete(ait.__anext__())
            except StopAsyncIteration:
                break
            else:
                yield item


mlflow.langchain.autolog()
agent = LangGraphResponsesAgent(langgraph_agent)
set_model(agent)
'''

agent_path = pathlib.Path("agent.py")
agent_path.write_text(agent_code)
print(f"Wrote agent code to {agent_path.resolve()}")

# Declare the FM API endpoint as a resource dependency
resources = [DatabricksServingEndpoint(endpoint_name=FM_API_ENDPOINT)]

input_example = {
    "input": [{"role": "user", "content": "What is Databricks?"}]
}

# Strip backticks for MLflow API (backticks are only needed for Spark SQL with hyphenated names)
uc_model_name_clean = UC_MODEL_NAME.replace("`", "")

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="langgraph-agent",
        python_model=str(agent_path),
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            "mlflow",
            "databricks-agents",
            "databricks-langchain",
            "langgraph",
            "nest_asyncio",
        ],
    )

print(f"Model logged: {model_info.model_uri}")

# Register to Unity Catalog
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=uc_model_name_clean,
)
print(f"Registered model: {registered_model.name} (version {registered_model.version})")

In [0]:
from databricks import agents
from databricks.sdk import WorkspaceClient
import datetime

# --- Deployment Configuration ---
# Adjust these parameters to control endpoint sizing, scaling, and metadata.

# Workload size: "Small", "Medium", or "Large"
# Determines the compute resources allocated per replica.
WORKLOAD_SIZE = "Small"

# Scale to zero when idle (saves cost, but adds cold-start latency on first request)
SCALE_TO_ZERO = True

# Custom endpoint name (None = auto-generated from model name)
ENDPOINT_NAME = "langgraph-agent-gpt-5-4"

# Environment variables passed to the serving container (e.g., feature flags, config)
ENVIRONMENT_VARS = {}

# Tags for tracking / filtering endpoints
TAGS = {
    "team": "data-science",
    "project": "langgraph-agent-demo",
}

# Endpoint description shown in the Serving UI
DESCRIPTION = "LangGraph ReAct agent with GPT 5.4 via FM API \u2014 streaming & non-streaming"

# Serverless usage policy ID (None = no policy; set once on first deploy, ignored on updates)
USAGE_POLICY_ID = None

# Max time to wait for the endpoint to become ready (minutes)
POLL_TIMEOUT_MINUTES = 30

# --- Deploy ---
uc_model_name_clean = UC_MODEL_NAME.replace("`", "")

deployment = agents.deploy(
    model_name=uc_model_name_clean,
    model_version=registered_model.version,
    workload_size=WORKLOAD_SIZE,
    scale_to_zero=SCALE_TO_ZERO,
    endpoint_name=ENDPOINT_NAME,
    environment_vars=ENVIRONMENT_VARS,
    tags=TAGS,
    description=DESCRIPTION,
    usage_policy_id=USAGE_POLICY_ID,
)

print(f"Endpoint name    : {deployment.endpoint_name}")
print(f"Endpoint URL     : {deployment.endpoint_url}")
print(f"Workload size    : {WORKLOAD_SIZE}")
print(f"Scale to zero    : {SCALE_TO_ZERO}")

# --- Poll until endpoint is ready ---
print(f"\nWaiting for endpoint to reach READY state (timeout: {POLL_TIMEOUT_MINUTES}m)...")

w = WorkspaceClient()

def _progress_callback(endpoint):
    ready = endpoint.state.ready if endpoint.state else "UNKNOWN"
    config = endpoint.state.config_update if endpoint.state else "UNKNOWN"
    print(f"  ready={ready}  config_update={config}")

endpoint_details = w.serving_endpoints.wait_get_serving_endpoint_not_updating(
    name=deployment.endpoint_name,
    timeout=datetime.timedelta(minutes=POLL_TIMEOUT_MINUTES),
    callback=_progress_callback,
)

final_state = endpoint_details.state.ready
print(f"\nEndpoint '{deployment.endpoint_name}' is now: {final_state}")
assert final_state.value == "READY", f"Endpoint not ready \u2014 state: {final_state}"

---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-5725228422353181>, line 74
     72 final_state = endpoint_details.state.ready
     73 print(f"\nEndpoint '{deployment.endpoint_name}' is now: {final_state}")
---> 74 assert str(final_state) == "READY", f"Endpoint not ready — state: {final_state}"

AssertionError: Endpoint not ready — state: EndpointStateReady.READY

In [0]:
import requests
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint_name = deployment.endpoint_name
base_url = f"{w.config.host}/serving-endpoints/{endpoint_name}/invocations"

# Get API token from notebook context (w.config.token is None on serverless compute)
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
payload = {
    "input": [{"role": "user", "content": "What is Unity Catalog in Databricks?"}]
}

# --- Non-Streaming ---
print("=== Non-Streaming ===")
resp = requests.post(base_url, headers=headers, json=payload)
resp.raise_for_status()
result = resp.json()
for item in result.get("output", []):
    if item.get("type") == "message":
        print(f"  Assistant: {item['content'][0]['text']}")
    elif item.get("type") == "function_call":
        print(f"  Tool call: {item['name']}({item['arguments']})")
print()

# --- Streaming ---
print("=== Streaming ===")
stream_payload = {**payload, "stream": True}
resp = requests.post(base_url, headers=headers, json=stream_payload, stream=True)
resp.raise_for_status()
for line in resp.iter_lines():
    if line:
        decoded = line.decode("utf-8")
        if decoded.startswith("data: "):
            data = decoded[len("data: "):]
            if data.strip() == "[DONE]":
                break
            event = json.loads(data)
            etype = event.get("type", "")
            if etype == "response.output_text.delta":
                print(event.get("delta", ""), end="", flush=True)
            elif etype == "response.output_item.done":
                item = event.get("item", {})
                if item.get("type") == "function_call":
                    print(f"\n  Tool call: {item['name']}({item['arguments']})")
print("\n--- stream complete ---")